# Notebook 12 — Lean Model v2 (Brand+Year IQR)

Retrains the lean quantile model using the improved `_remove_outliers_iqr` that groups by `(brand, year)` instead of `brand` only.

**Key changes vs v1 (NB06/NB07):**
- IQR computed per `(brand, year)` → old cars compared against their own cohort  
- ±1 year pooling for groups < 50 cars  
- Groups with < 10 cars after pooling are dropped (printed during cleaning)  

**Sections:**
1. Setup & Constants  
2. Load & Clean (new IQR)  
3. Train / Test Split  
4. Feature Engineering (lean spec)  
5. Train Quantile Models (q15 / q50 / q85)  
6. MLflow Logging  
7. Save Models  
8. Internal Evaluation by Year  
9. Customs Evaluation — CI & CM  

## 1. Setup & Constants

In [ ]:
import os
import sys
import json
import importlib
import joblib
from datetime import datetime
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
from lightgbm import LGBMRegressor
import mlflow
import mlflow.lightgbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Project root
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import data_processing
import features.feature_engineering as _fe_module
importlib.reload(_fe_module)
from features.feature_engineering import CarPriceFeatureEngineer
from data_processing import CarDataProcessor
from config import DATA_PATH

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# ── Constants ────────────────────────────────────────────────────────────────
CURRENT_YEAR     = 2025
TRAIN_TEST_SEED  = 42
MLFLOW_URI       = 'http://localhost:5001'
VARIANT          = 'lean_v2_brand_year_iqr'

data_dir   = Path(os.path.join(DATA_PATH, 'le_boncoin_13_oct_2025'))
models_dir = project_root / 'models' / 'lean_quantile_v2'
output_dir = project_root / 'output'

print(f'Project root : {project_root}')
print(f'Data dir     : {data_dir}')
print(f'Models dir   : {models_dir}')
print(f'MLflow URI   : {MLFLOW_URI}')
print('✓ All imports successful')

## 2. Load & Clean (Brand+Year IQR)

In [ ]:
# Load raw data
df_raw = data_processing.load_car_data(data_dir)
print(f'Raw rows: {len(df_raw):,}')

# Clean with new brand+year IQR (large_group_threshold=50, min_group_size=10)
processor = CarDataProcessor(
    large_group_threshold=50,
    min_group_size=10,
    verbose=True,
)
df = processor.clean_data(df_raw)

# Add log_price (plain log, matching NB07/NB09 convention)
df = df.with_columns(pl.col('price').log().alias('log_price'))

print(f'\n━━━ Cleaning summary ━━━')
stats = processor.cleaning_stats.get('outlier_removal', {})
print(f"  Rows before IQR step  : {stats.get('rows_before', '?'):,}")
print(f"  Removed by IQR        : {stats.get('n_dropped_iqr', '?'):,}")
print(f"  Removed (tiny groups) : {stats.get('n_dropped_small_groups', '?'):,}")
print(f"  Rows after            : {stats.get('rows_after', '?'):,}")
print(f'\nFinal dataset: {len(df):,} rows')
print(f'Year range   : {df["year"].min()} – {df["year"].max()}')
print(f'Unique brands: {df["brand"].n_unique()}')
print(f'Unique models: {df["model"].n_unique()}')

## 3. Train / Test Split

In [ ]:
train_indices, test_indices = train_test_split(
    range(len(df)), test_size=0.2, random_state=TRAIN_TEST_SEED
)
df_train = df[train_indices]
df_test  = df[test_indices]

# Arrays for grouped error metrics later
years_test   = df_test['year'].to_numpy()
car_age_test = CURRENT_YEAR - years_test
y_test_eur   = df_test['price'].to_numpy()

print(f'Train: {len(df_train):,}  |  Test: {len(df_test):,}')
print(f'Test year range : {years_test.min()} – {years_test.max()}')
print(f'Test price range: €{y_test_eur.min():,.0f} – €{y_test_eur.max():,.0f}')

## 4. Feature Engineering — Lean Spec

Fit on training data only (no leakage). Lean spec: no OHE, no km, no HP, no energie.

In [ ]:
feature_engineer = CarPriceFeatureEngineer(
    current_year=CURRENT_YEAR,
    brand_onehot=False,
    model_onehot=False,
    add_horsepower_features=False,
    add_energie_ohe=False,
)

# Fit on training set ONLY
feature_engineer.fit(
    df_train.drop(['price', 'log_price']),
    df_train['price'],
)

# Transform both sets
df_train_fe = feature_engineer.transform(df_train.drop(['price', 'log_price']))
df_test_fe  = feature_engineer.transform(df_test.drop(['price', 'log_price']))

feat_cols = df_train_fe.columns
X_train = df_train_fe.to_pandas().fillna(0)
X_test  = df_test_fe.to_pandas().fillna(0)
y_train = df_train['log_price'].to_numpy()
y_test  = df_test['log_price'].to_numpy()

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Features: {feat_cols[:8].to_list()} ... ({len(feat_cols)} total)')

## 5. Train Quantile Models (q15 / q50 / q85)

In [ ]:
LGB_PARAMS = dict(
    objective='quantile',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=TRAIN_TEST_SEED,
    n_jobs=-1,
    verbose=-1,
)

models = {}
preds_log = {}
preds_eur = {}

for alpha, label in [(0.15, 'q15'), (0.50, 'q50'), (0.85, 'q85')]:
    print(f'Training {label} (alpha={alpha})…', end=' ', flush=True)
    m = LGBMRegressor(alpha=alpha, **LGB_PARAMS)
    m.fit(X_train, y_train)
    preds_log[label] = m.predict(X_test)
    preds_eur[label] = np.exp(preds_log[label])
    models[label]    = m
    print('done ✓')

print(f'\nq50 MAE : €{mean_absolute_error(y_test_eur, preds_eur["q50"]):,.0f}')
coverage_q1585 = float(np.mean(
    (preds_eur['q15'] <= y_test_eur) & (y_test_eur <= preds_eur['q85'])
))
print(f'Q15-Q85 coverage: {coverage_q1585*100:.1f}%  (target ~70%)')

## 6. MLflow Logging

In [ ]:
def pinball_loss(y_true, y_pred, alpha):
    """Quantile (pinball) loss — primary metric for quantile models."""
    errors = y_true - y_pred
    return float(np.mean(np.where(errors >= 0, alpha * errors, (alpha - 1) * errors)))

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('car_price_lean_v2')

with mlflow.start_run(run_name=VARIANT) as run:
    mlflow.log_params({
        'variant':               VARIANT,
        'n_train':               len(X_train),
        'n_test':                len(X_test),
        'n_features':            len(feat_cols),
        'large_group_threshold': 50,
        'min_group_size':        10,
        'lgb_n_estimators':      LGB_PARAMS['n_estimators'],
        'lgb_learning_rate':     LGB_PARAMS['learning_rate'],
        'lgb_num_leaves':        LGB_PARAMS['num_leaves'],
    })

    for alpha, label in [(0.15, 'q15'), (0.50, 'q50'), (0.85, 'q85')]:
        pb = pinball_loss(y_test, preds_log[label], alpha)
        mlflow.log_metric(f'pinball_{label}', pb)
        print(f'  pinball_{label}: {pb:.5f}')

    mlflow.log_metric('coverage_q15_q85', coverage_q1585)
    mlflow.log_metric('mae_q50_eur',      mean_absolute_error(y_test_eur, preds_eur['q50']))

    # Feature importances as artifact
    fi = pd.DataFrame({
        'feature':    feat_cols,
        'importance': models['q50'].feature_importances_,
    }).sort_values('importance', ascending=False)
    fi_path = '/tmp/feature_importances_v2.csv'
    fi.to_csv(fi_path, index=False)
    mlflow.log_artifact(fi_path)

    run_id = run.info.run_id

print(f'\n✓ MLflow run logged  (run_id={run_id})')
print(f'  View: {MLFLOW_URI}')

## 7. Save Models

In [ ]:
models_dir.mkdir(parents=True, exist_ok=True)

for label, m in models.items():
    path = models_dir / f'lgb_{label}_lean.pkl'
    joblib.dump(m, path)
    print(f'  Saved {path.name}')

joblib.dump(feature_engineer, models_dir / 'feature_engineer_lean.pkl')
print('  Saved feature_engineer_lean.pkl')

metadata = {
    'training_date':         datetime.now().isoformat(),
    'variant':               VARIANT,
    'n_samples':             len(df_train),
    'n_features':            len(feat_cols),
    'mlflow_run_id':         run_id,
    'cleaning': {
        'large_group_threshold': 50,
        'min_group_size':        10,
    },
    'lgb_params':            LGB_PARAMS,
    'validation_mae_q50':    mean_absolute_error(y_test_eur, preds_eur['q50']),
    'validation_coverage':   coverage_q1585 * 100,
    'features': {
        'all_features': feat_cols.to_list(),
    },
}

with open(models_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('  Saved metadata.json')

all_brands = list(feature_engineer.brand_price_stats_.keys())
all_models_list = list(feature_engineer.model_price_stats_.keys())
(models_dir / 'brand_list.txt').write_text('\n'.join(sorted(all_brands)))
(models_dir / 'feature_list.txt').write_text('\n'.join(feat_cols))
print(f'\n✓ All artifacts saved to {models_dir}')
print(f'  Known brands: {len(all_brands)}  |  Known models: {len(all_models_list)}')

## 8. Internal Evaluation by Year

Primary metric: `pct_err_over_year_median = mean(|actual − pred_q50|) / median(actual prices for that year) × 100`

Compare old-car error vs v1 baseline from NB11.

In [ ]:
AGE_BUCKETS = [
    (0,  2,  '0-2 yr (2023-25)'),
    (3,  5,  '3-5 yr (2020-22)'),
    (6,  10, '6-10 yr (2015-19)'),
    (11, 15, '11-15 yr (2010-14)'),
    (16, 20, '16-20 yr (2005-09)'),
    (21, 99, '21+ yr (pre-2005)'),
]
AGE_BUCKET_LABELS = [lab for _, _, lab in AGE_BUCKETS]

def age_to_bucket(age):
    for lo, hi, lab in AGE_BUCKETS:
        if lo <= age <= hi:
            return lab
    return 'unknown'


def compute_year_metrics(y_true, y_q50, y_q15, y_q85, groups, group_col='year'):
    rows = []
    for gval in sorted(np.unique(groups)):
        mask = groups == gval
        if mask.sum() < 3:
            continue
        act, pred = y_true[mask], y_q50[mask]
        lo, hi    = y_q15[mask], y_q85[mask]
        med = np.median(act)
        rows.append({
            group_col:              gval,
            'n':                    int(mask.sum()),
            'median_price_eur':     round(med, 0),
            'pct_err_over_median':  round(100 * mean_absolute_error(act, pred) / med, 2),
            'mae_eur':              round(mean_absolute_error(act, pred), 0),
            'median_signed_pct':    round(float(np.median((pred - act) / np.clip(act, 1, None) * 100)), 2),
            'coverage_70':          round(float(np.mean((lo <= act) & (act <= hi))), 4),
        })
    return pd.DataFrame(rows)


year_metrics   = compute_year_metrics(y_test_eur, preds_eur['q50'], preds_eur['q15'], preds_eur['q85'], years_test)
bucket_labels  = np.array([age_to_bucket(a) for a in car_age_test])
bucket_metrics = compute_year_metrics(y_test_eur, preds_eur['q50'], preds_eur['q15'], preds_eur['q85'], bucket_labels, 'year_bucket')
bucket_metrics['_order'] = bucket_metrics['year_bucket'].map({lab: i for i, (_, _, lab) in enumerate(AGE_BUCKETS)})
bucket_metrics = bucket_metrics.sort_values('_order').drop(columns='_order').reset_index(drop=True)

print('=== By year (first 10 rows) ===')
print(year_metrics.head(10).to_string(index=False))
print('\n=== By age bucket ===')
print(bucket_metrics.to_string(index=False))

In [ ]:
# ── Chart: pct_err_over_median by year ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(year_metrics['year'], year_metrics['pct_err_over_median'],
        marker='o', linewidth=2.0, color='#4C72B0', markersize=6, label=VARIANT)
for _, row in year_metrics.iterrows():
    ax.annotate(f"n={int(row['n'])}",
                (row['year'], row['pct_err_over_median']),
                textcoords='offset points', xytext=(0, 8), fontsize=7, ha='center', color='grey')
ax.axhline(year_metrics['pct_err_over_median'].mean(), color='crimson',
           linestyle='--', linewidth=1.2, label='Overall mean')
ax.set_xlabel('Car Year')
ax.set_ylabel('% Error over Year Median Price')
ax.set_title(f'[{VARIANT}] Normalised Error by Calendar Year', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart: normalised error by age bucket ────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(14, 5))
x = range(len(bucket_metrics))
bars = ax1.bar(x, bucket_metrics['pct_err_over_median'], color='#4C72B0', width=0.6, alpha=0.85)
ax1.set_xticks(list(x))
ax1.set_xticklabels(bucket_metrics['year_bucket'], fontsize=10)
ax1.set_ylabel('% Error over Age-Bucket Median Price', fontsize=11)
ax1.set_title(f'[{VARIANT}] Normalised Error by Car Age Bracket', fontsize=12, fontweight='bold')
for bar, val in zip(bars, bucket_metrics['pct_err_over_median']):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax2 = ax1.twinx()
ax2.plot(list(x), bucket_metrics['n'], color='darkorange', marker='D', linewidth=2, markersize=7, label='n (test)')
ax2.set_ylabel('Test set sample count', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')
ax2.legend(loc='upper right')
wm = (bucket_metrics['pct_err_over_median'] * bucket_metrics['n']).sum() / bucket_metrics['n'].sum()
ax1.axhline(wm, color='crimson', linestyle='--', linewidth=1.2, label=f'Weighted mean {wm:.1f}%')
ax1.legend(loc='upper left')
ax1.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart: Q15-Q85 coverage by year ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(year_metrics['year'], year_metrics['coverage_70'] * 100,
        marker='s', linewidth=2.0, color='#55A868', markersize=6)
ax.axhline(70, color='crimson', linestyle='--', linewidth=1.2, label='Ideal 70% coverage')
ax.set_ylim(0, 110)
ax.set_xlabel('Car Year')
ax.set_ylabel('Coverage (%)')
ax.set_title(f'[{VARIANT}] Q15–Q85 Interval Coverage by Year', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 9. Customs Evaluation — CI & CM

Loads the same customs Excel files as NB07, applies the same brand/model corrections, then predicts with the new v2 models.

In [ ]:
# ── Load CI ───────────────────────────────────────────────────────────────────
ci_path = '/Users/brunobrumbrum/Downloads/VEHICULE_CHASSIS (1).xlsx'

df_ci_raw = pl.read_excel(ci_path)
df_ci = (
    df_ci_raw
    .with_columns([
        pl.col('MARQUE').str.to_lowercase().str.strip_chars()
            .str.replace('land rover', 'land-rover', literal=True).alias('brand'),
        pl.col('MODELE').str.to_lowercase().str.strip_chars().alias('model'),
        pl.col('PREMIERE_MIS_CIRCULAT').str.slice(0, 4).cast(int).alias('year'),
        (pl.col('VALCAF') * 0.0015).alias('actual_price_eur'),
        pl.lit(100000).alias('km'),
        pl.lit('CI').alias('country'),
    ])
    .with_columns([
        pl.col('brand').replace('land-rover', 'landrover')
                       .replace('mercedes-benz', 'mercedesbenz').alias('brand'),
        pl.col('model').replace({
            'cr-v': 'crv', 'rogue': 'xtrail',
            'mirage': 'space star', 'glc': 'classe glc', 'tacoma': 'hilux',
        }).alias('model'),
    ])
)
print(f'CI raw: {len(df_ci):,} vehicles')

# ── Load CM ───────────────────────────────────────────────────────────────────
cm_path = '/Users/brunobrumbrum/Downloads/Extraction_Jeu_données_Test_Vehicules (2).xlsx'

df_cm_raw = pl.read_excel(cm_path)
df_cm = (
    df_cm_raw
    .with_columns([
        pl.col('Marque').str.to_lowercase().str.strip_chars().alias('brand'),
        pl.col('Modèle').str.to_lowercase().str.strip_chars().alias('model'),
        pl.col('Année Fabrication').cast(pl.Int64).alias('year'),
        (pl.col('Valeur imposable (XAF)') * 0.0015).alias('actual_price_eur'),
        pl.lit(100000).alias('km'),
        pl.lit('CM').alias('country'),
    ])
    .with_columns([
        pl.col('brand').replace({'land rover': 'landrover'}).alias('brand'),
        pl.col('model').replace({
            'rav 4': 'rav4', 'toyota corolla': 'corolla',
            'toyota yaris verso': 'yaris verso', 'c-max': 'cmax', 'carina': 'avensis',
        }).alias('model'),
    ])
)
print(f'CM raw: {len(df_cm):,} vehicles')

In [ ]:
# ── Filter to brands/models known to the NEW feature engineer ─────────────────

def filter_and_predict(df_customs, fe, model_q15, model_q50, model_q85, feat_cols, label):
    known_brands = set(fe.brand_price_stats_.keys())
    known_models = set(fe.model_price_stats_.keys())

    before = len(df_customs)
    df_clean = df_customs.filter(
        pl.col('brand').is_in(list(known_brands))
        & pl.col('model').is_in(list(known_models))
    )
    after = len(df_clean)
    print(f'{label}: {before:,} → {after:,} vehicles ({before-after:,} removed, unknown brand/model)')

    # Feature engineering
    df_fe = fe.transform(df_clean.select(['brand', 'model', 'year', 'km']))
    X = df_fe.to_pandas().fillna(0)
    for c in feat_cols:
        if c not in X.columns:
            X[c] = 0
    X = X[feat_cols]

    # Predict (log-space → EUR)
    p15 = np.exp(model_q15.predict(X))
    p50 = np.exp(model_q50.predict(X))
    p85 = np.exp(model_q85.predict(X))

    df_result = df_clean.with_columns([
        pl.Series('pred_q15', p15),
        pl.Series('pred_q50', p50),
        pl.Series('pred_q85', p85),
        pl.Series('residual', df_clean['actual_price_eur'].to_numpy() - p50),
        pl.Series('pct_error', (df_clean['actual_price_eur'].to_numpy() - p50) / p50 * 100),
    ])
    return df_result


df_ci_results = filter_and_predict(
    df_ci, feature_engineer, models['q15'], models['q50'], models['q85'],
    feat_cols.to_list(), 'CI'
)
df_cm_results = filter_and_predict(
    df_cm, feature_engineer, models['q15'], models['q50'], models['q85'],
    feat_cols.to_list(), 'CM'
)

In [ ]:
# ── Performance metrics ───────────────────────────────────────────────────────

def calc_metrics(df_res):
    act   = df_res['actual_price_eur'].to_numpy()
    p15   = df_res['pred_q15'].to_numpy()
    p50   = df_res['pred_q50'].to_numpy()
    p85   = df_res['pred_q85'].to_numpy()
    return {
        'n':                   len(act),
        'MAE (€)':             round(mean_absolute_error(act, p50), 0),
        'MAPE (%)':            round(np.mean(np.abs((act - p50) / act)) * 100, 1),
        'R²':                  round(r2_score(act, p50), 3),
        'Coverage Q15-Q85 (%)': round(np.mean((act >= p15) & (act <= p85)) * 100, 1),
        'Median declared (€)': round(float(np.median(act)), 0),
        'Median pred_q50 (€)': round(float(np.median(p50)), 0),
    }

metrics_ci = calc_metrics(df_ci_results)
metrics_cm = calc_metrics(df_cm_results)

metrics_df = pd.DataFrame({
    'Metric': list(metrics_ci.keys()),
    "Côte d'Ivoire (CI)": list(metrics_ci.values()),
    'Cameroon (CM)':      list(metrics_cm.values()),
})

print('\n' + '='*60)
print('CUSTOMS EVALUATION METRICS')
print('='*60)
print(metrics_df.to_string(index=False))

In [ ]:
# ── Actual vs predicted scatter — CI & CM ────────────────────────────────────
colors = {'CI': 'green', 'CM': 'orange'}
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (country, df_res) in zip(axes, [('CI', df_ci_results), ('CM', df_cm_results)]):
    act = df_res['actual_price_eur'].to_numpy()
    p15 = df_res['pred_q15'].to_numpy()
    p50 = df_res['pred_q50'].to_numpy()
    p85 = df_res['pred_q85'].to_numpy()
    sort_idx = np.argsort(p50)

    ax.fill_between(range(len(sort_idx)), p15[sort_idx], p85[sort_idx],
                    alpha=0.3, color=colors[country], label='Q15-Q85')
    ax.plot(range(len(sort_idx)), p50[sort_idx],
            color=colors[country], linewidth=1.8, label='Pred Q50')

    inside = (act >= p15) & (act <= p85)
    ax.scatter(np.arange(len(sort_idx))[inside[sort_idx]],
               act[sort_idx][inside[sort_idx]],
               color='darkgreen', s=20, alpha=0.7, label='Inside CI')
    ax.scatter(np.arange(len(sort_idx))[~inside[sort_idx]],
               act[sort_idx][~inside[sort_idx]],
               color='red', marker='x', s=25, alpha=0.7, label='Outside CI')

    cov = np.mean(inside) * 100
    cname = "Côte d'Ivoire" if country == 'CI' else 'Cameroon'
    ax.set_title(f'{cname} ({country})  —  Coverage: {cov:.1f}%', fontsize=12)
    ax.set_xlabel('Vehicle (sorted by pred Q50)')
    ax.set_ylabel('Price (€)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'€{v:,.0f}'))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'[{VARIANT}] Actual vs Predicted — Customs Data', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Validation plots (ported from NB07) ───────────────────────────────────────
WARN_SUFFIX = ' [!]'

def prepare_validation_data(df_customs, df_training, year_flex=2):
    results = []
    df_train_pd = df_training.to_pandas()
    for row in df_customs.iter_rows(named=True):
        brand, model, year  = row['brand'], row['model'], row['year']
        declared            = row['actual_price_eur']
        p15, p50, p85       = row['pred_q15'], row['pred_q50'], row['pred_q85']
        mask = (
            (df_train_pd['brand'] == brand)
            & (df_train_pd['model'] == model)
            & (df_train_pd['year'] >= year - year_flex)
            & (df_train_pd['year'] <= year + year_flex)
        )
        matched = df_train_pd[mask]
        n_match = len(matched)
        if n_match > 0:
            prices = matched['price'].values
            tp10, tq15, tq50, tq85, tp90 = (
                float(np.percentile(prices, 10)), float(np.percentile(prices, 15)),
                float(np.percentile(prices, 50)), float(np.percentile(prices, 85)),
                float(np.percentile(prices, 90)),
            )
            has_data = True
        else:
            tp10 = tq15 = tq50 = tq85 = tp90 = np.nan
            has_data = False
        if p15 <= declared <= p85:
            tier = 'GREEN'
        elif has_data and (tp10 <= declared <= tp90):
            tier = 'ORANGE'
        else:
            tier = 'RED'
        results.append({
            'brand': brand, 'model': model, 'year': year,
            'declared_price': declared,
            'pred_q15': p15, 'pred_q50': p50, 'pred_q85': p85,
            'train_p10': tp10, 'train_q15': tq15, 'train_q50': tq50,
            'train_q85': tq85, 'train_p90': tp90,
            'gap_pct': (p50 - declared) / p50 * 100,
            'color_tier': tier,
            'training_sample_size': n_match,
            'has_training_data': has_data,
        })
    return pd.DataFrame(results).sort_values(['brand', 'model', 'year']).reset_index(drop=True)


def create_validation_plot(df_subset, title, median_declared):
    n_cars = len(df_subset)
    if n_cars == 0:
        return None
    fig, ax = plt.subplots(figsize=(16, max(6, n_cars * 0.45)), dpi=120)
    cmap = {'GREEN': 'green', 'ORANGE': 'orange', 'RED': 'red'}
    labels = [
        f"{r['brand']} {r['model']} {int(r['year'])}" + (WARN_SUFFIX if r['training_sample_size'] < 5 else '')
        for _, r in df_subset.iterrows()
    ]
    yp = np.arange(n_cars)
    for i, (_, row) in enumerate(df_subset.iterrows()):
        y = yp[i]
        if row['has_training_data'] and not np.isnan(row['train_p10']):
            ax.fill_betweenx([y-.35, y+.35], row['train_p10'], row['train_p90'],
                             color='lightgrey', alpha=0.35, zorder=1)
            ax.vlines(row['train_q50'], y-.30, y+.30, color='#444444', linewidth=2, zorder=2)
            ax.vlines(row['train_q15'], y-.15, y+.15, color='#888888', linewidth=1.5, zorder=2)
            ax.vlines(row['train_q85'], y-.15, y+.15, color='#888888', linewidth=1.5, zorder=2)
        ax.hlines(y, row['pred_q15'], row['pred_q85'], colors='steelblue', linewidth=3, alpha=0.5, zorder=3)
        ax.vlines(row['pred_q50'], y-.25, y+.25, color='navy', linewidth=2.5, zorder=4)
        ax.vlines(row['pred_q15'], y-.20, y+.20, color='steelblue', linewidth=2, zorder=4)
        ax.vlines(row['pred_q85'], y-.20, y+.20, color='steelblue', linewidth=2, zorder=4)
        ax.scatter(row['declared_price'], y, color=cmap[row['color_tier']],
                   s=60, zorder=5, edgecolors='black', linewidths=0.5)
    ax.set_xlim(auto=True)
    xmi, xma = ax.get_xlim()
    for i, (_, row) in enumerate(df_subset.iterrows()):
        g = row['gap_pct']
        ax.annotate(f"{'+' if g>0 else ''}{g:.1f}%",
                    xy=(xma - (xma-xmi)*0.02, yp[i]),
                    fontsize=10, color=cmap[row['color_tier']], va='center', ha='right')
    ax.axvline(median_declared, color='purple', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.annotate(f'Median declared\n€{median_declared:,.0f}',
                xy=(median_declared, n_cars - 0.5),
                fontsize=10, color='purple', ha='center', va='bottom')
    ax.set_yticks(yp)
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xlabel('Price (€)', fontsize=12)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'€{v:,.0f}'))
    ax.set_title(title, fontsize=12, pad=10)
    ax.grid(True, axis='x', alpha=0.3)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    return fig


print('Preparing validation data…')
df_val_ci = prepare_validation_data(df_ci_results, df, year_flex=1)
df_val_cm = prepare_validation_data(df_cm_results, df, year_flex=2)

# Save CSVs
output_dir.mkdir(exist_ok=True)
csv_cols = ['brand', 'model', 'year', 'declared_price',
            'pred_q15', 'pred_q50', 'pred_q85',
            'train_p10', 'train_q15', 'train_q50', 'train_q85', 'train_p90',
            'gap_pct', 'color_tier', 'training_sample_size']
df_val_ci[csv_cols].to_csv(output_dir / 'validation_CI_v2.csv', index=False)
df_val_cm[csv_cols].to_csv(output_dir / 'validation_CM_v2.csv', index=False)
print(f'✓ Validation CSVs saved to {output_dir}')

# Tier summary
for country, df_v in [('CI', df_val_ci), ('CM', df_val_cm)]:
    tiers = df_v['color_tier'].value_counts()
    print(f"  {country}: GREEN={tiers.get('GREEN',0)}  ORANGE={tiers.get('ORANGE',0)}  RED={tiers.get('RED',0)}")

In [ ]:
# ── Validation plots grouped by brand ────────────────────────────────────────
def plot_groups(df_val, country_code, country_name, brand_groups, year_flex):
    print(f'\n{"+"*60}')
    print(f'VALIDATION PLOTS — {country_name.upper()} ({country_code})')
    print(f'{"+"*60}')
    for group_label, brands in brand_groups.items():
        sub = df_val[df_val['brand'].isin(brands)].copy()
        if sub.empty:
            print(f'  Skipping "{group_label}" — no vehicles')
            continue
        n_per_tier = sub['color_tier'].value_counts()
        print(f'  "{group_label}": {len(sub)} vehicles  '
              f'GREEN={n_per_tier.get("GREEN",0)}  '
              f'ORANGE={n_per_tier.get("ORANGE",0)}  '
              f'RED={n_per_tier.get("RED",0)}')
        title = f'{country_name} — {group_label} (year ±{year_flex}) [{VARIANT}]'
        create_validation_plot(sub, title, sub['declared_price'].median())


# ── CI brand groups ───────────────────────────────────────────────────────────
ci_brand_groups = OrderedDict([
    ('Chevrolet · Citroën · Peugeot · Ford', ['chevrolet', 'citroen', 'citroën', 'peugeot', 'ford']),
    ('Honda · Hyundai',                      ['honda', 'hyundai']),
    ('Jeep · Land Rover · Mercedes',         ['jeep', 'landrover', 'mercedesbenz', 'mercedes', 'mercedes-benz']),
    ('Kia · Mazda · Mitsubishi · Nissan',    ['kia', 'mazda', 'mitsubishi', 'nissan']),
    ('Toyota',                               ['toyota']),
])
plot_groups(df_val_ci.drop_duplicates(), 'CI', "Ivory Coast", ci_brand_groups, year_flex=1)

# ── CM brand groups ───────────────────────────────────────────────────────────
all_cm_brands = sorted(df_val_cm['brand'].unique())
cm_brand_groups = OrderedDict()
non_toyota = [b for b in all_cm_brands if b != 'toyota']
if non_toyota:
    cm_brand_groups['All Brands (excl. Toyota)'] = non_toyota
if 'toyota' in all_cm_brands:
    cm_brand_groups['Toyota'] = ['toyota']
plot_groups(df_val_cm.drop_duplicates(), 'CM', 'Cameroon', cm_brand_groups, year_flex=2)

In [ ]:
print('='*60)
print('FINAL SUMMARY')
print('='*60)
print(f'Variant        : {VARIANT}')
print(f'Training rows  : {len(df_train):,}')
print(f'Test rows      : {len(df_test):,}')
print(f'Models saved to: {models_dir}')
print(f'MLflow run     : {run_id}')
print(f"\nInternal test metrics:")
print(f"  MAE (q50, €)        : {mean_absolute_error(y_test_eur, preds_eur['q50']):,.0f}")
print(f"  Q15-Q85 coverage    : {coverage_q1585*100:.1f}%")
print(f"\nCustoms metrics:")
for country, m in [('CI', metrics_ci), ('CM', metrics_cm)]:
    print(f"  {country}: MAE=€{m['MAE (€)']:,.0f}  MAPE={m['MAPE (%)']:.1f}%  "
          f"R²={m['R²']:.3f}  Coverage={m['Coverage Q15-Q85 (%)']:.1f}%")
print('\n✓ Notebook complete')